# Source CSV Patch Notebook (Stage 1)
Owns **all source-level patches**, kept apart from data prep. Reads raw CSVs from `RAW_DIR`,
writes the patched dataset to `PATCHED_DIR`. Data prep (Stage 2) reads only `PATCHED_DIR`.

**Patches applied**
1. Exclude `disclosures`, `data_requirements`, `disclosure_data_map` (dropped per decision).
2. Vehicles: recompute `scope1_tco2e_2024_clean` (fuel-based, EVs forced to 0) - idempotent.
3. Governance board-percentage snapping to whole directors - **optional**, off by default.
4. Presence check for tables added by historical patches - warns instead of silently regenerating.
5. Source FK integrity gate (bank_id and counterparty links).
6. Writes `_patch_log.json` provenance.

In [1]:
import os, json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\HP\Documents\IFRS_Reporting")

RAW_DATA_DIR = PROJECT_ROOT / "notebooks" / "gen_data" / "csv"

RAW_DIR      = RAW_DATA_DIR              # raw CSVs (from data.zip)
PATCHED_DIR  = PROJECT_ROOT / "notebooks" / "gen_data_patched"   # output of this notebook
APPLY_GOV_SNAP = False              # optional coherence fix; off = faithful to source
DROP_TABLES = {"disclosures", "data_requirements", "disclosure_data_map"}

os.makedirs(PATCHED_DIR, exist_ok=True)
for f in os.listdir(PATCHED_DIR):          # clean stale outputs
    if f.endswith((".csv", ".json")):
        os.remove(os.path.join(PATCHED_DIR, f))

tables, dropped = {}, []
for f in sorted(os.listdir(RAW_DIR)):
    if not f.endswith(".csv"):
        continue
    name = f[:-4]
    if name in DROP_TABLES:
        dropped.append(name); continue
    tables[name] = pd.read_csv(os.path.join(RAW_DIR, f))
print(f"loaded {len(tables)} tables | dropped (present in raw): {dropped or 'none'}")

loaded 30 tables | dropped (present in raw): ['data_requirements', 'disclosure_data_map', 'disclosures']


## Patch 1 - Vehicles: clean Scope 1 recalculation
Fuel-based recomputation (2.67 kg CO2/l), EVs forced to 0. Idempotent: recomputes the column even if present.

In [2]:
veh = tables["vehicles"].copy()
EMISSION_FACTOR = 2.67  # kg CO2 per litre, consistent across all fuel types

def recalc_scope1(row):
    if row["fuel_type"] == "electric":
        return 0.0                      # EVs have zero Scope 1 by definition
    if row["annual_fuel_consumption_l"] <= 0:
        return 0.0
    return round(row["annual_fuel_consumption_l"] * EMISSION_FACTOR / 1000, 6)

veh["scope1_tco2e_2024_clean"] = veh.apply(recalc_scope1, axis=1)

veh["delta_pct"] = (
    (veh["scope1_tco2e_2024_clean"] - veh["scope1_tco2e_2024"]) /
    veh["scope1_tco2e_2024"].replace(0, float("nan")) * 100
).abs()
print("EV rows forced to 0:", (veh["fuel_type"] == "electric").sum())
print("Rows still >1% delta (non-EV):",
      ((veh["delta_pct"] > 1) & (veh["fuel_type"] != "electric")).sum())
comparison = veh.groupby("bank_id").agg(
    original=("scope1_tco2e_2024", "sum"),
    recalculated=("scope1_tco2e_2024_clean", "sum")).round(2)
comparison["diff"] = (comparison["recalculated"] - comparison["original"]).round(2)
print(comparison)
tables["vehicles"] = veh

EV rows forced to 0: 62
Rows still >1% delta (non-EV): 0
         original  recalculated  diff
bank_id                              
BANK01     623.72        622.58 -1.14
BANK02     476.17        475.52 -0.65
BANK03     276.17        276.16 -0.01
BANK04      28.41         27.25 -1.16
BANK05     146.50        145.99 -0.51


## Patch 2 - Governance board-percentage snap (optional)
Snaps director-based percentages to whole-director equivalents. Off by default to keep source values untouched. `climate_on_board_agenda_pct` is meeting-based (recomputed downstream from board minutes) so it is deliberately not snapped here.

In [3]:
fix_log = []
if APPLY_GOV_SNAP:
    gov = tables["governance"].copy()
    SNAP_COLS = ["all_exec_climate_remuneration_pct",
                 "independent_directors_pct", "board_climate_expertise_pct"]
    for i, r in gov.iterrows():
        bs = r.get("board_size")
        if pd.isna(bs) or not bs:
            continue
        for col in SNAP_COLS:
            if col in gov.columns and pd.notna(r.get(col)):
                old = float(r[col])
                new = round(round(old / 100 * float(bs)) / float(bs) * 100, 1)
                if new != old:
                    gov.at[i, col] = new
                    fix_log.append({"table": "governance", "bank_id": r["bank_id"],
                                    "field": col, "old": old, "new": new,
                                    "reason": f"snapped to whole directors of board_size={int(bs)}"})
    tables["governance"] = gov
    print(f"governance snap applied: {len(fix_log)} fixes")
else:
    print("governance snap OFF - source values kept as-is")

governance snap OFF - source values kept as-is


## Patch 3 - Historical-patch presence checks
`climate_opportunities.csv`, scenario enrichment columns, and the six IFRS S2 tables were added by earlier patch versions and now ship inside the raw data. Verify presence instead of regenerating.

In [4]:
warnings = []
if "climate_opportunities" not in tables:
    warnings.append("climate_opportunities.csv MISSING - re-run historical patch 2.4.2")
cs = tables.get("climate_scenarios")
for col in ("methodology_notes", "resilience_assessment"):
    if cs is not None and col not in cs.columns:
        warnings.append(f"climate_scenarios missing enrichment column '{col}' (patch 2.4.3)")
for t in ("scope3_categories","ghg_methodology","scope12_consolidation",
          "transition_plan","resilience_assessment","climate_financial_effects"):
    if t not in tables:
        warnings.append(f"IFRS S2 table '{t}' missing from raw data")
print("\n".join(warnings) if warnings else "all expected patched tables/columns present")

all expected patched tables/columns present


## Patch 4 - Source FK integrity gate
Relationship checks on the source data (bank and counterparty links). Reports orphans; does not mutate.

In [5]:
def check_fk(child, child_col, parent, parent_col, child_name, parent_name):
    if child_col not in child.columns or parent_col not in parent.columns:
        return
    orphans = set(child[child_col].dropna()) - set(parent[parent_col].dropna())
    status = "OK " if not orphans else f"ORPHANS({len(orphans)})"
    print(f"  {status} {child_name}.{child_col} -> {parent_name}.{parent_col}")

banks = tables["banks"]
print("bank_id links:")
for name, df in tables.items():
    if name != "banks" and "bank_id" in df.columns:
        check_fk(df, "bank_id", banks, "bank_id", name, "banks")
print("counterparty links:")
for name in ("exposures", "counterparty_emissions", "collateral", "investments"):
    if name in tables and "counterparty_id" in tables[name].columns:
        check_fk(tables[name], "counterparty_id", tables["counterparties"],
                 "counterparty_id", name, "counterparties")

bank_id links:
  OK  board_minutes_extract.bank_id -> banks.bank_id
  OK  carbon_credits.bank_id -> banks.bank_id
  OK  climate_financial_effects.bank_id -> banks.bank_id
  OK  climate_opportunities.bank_id -> banks.bank_id
  OK  climate_risk_register.bank_id -> banks.bank_id
  OK  climate_scenarios.bank_id -> banks.bank_id
  OK  collateral.bank_id -> banks.bank_id
  OK  counterparties.bank_id -> banks.bank_id
  OK  employees.bank_id -> banks.bank_id
  OK  exposures.bank_id -> banks.bank_id
  OK  facilities.bank_id -> banks.bank_id
  OK  financial_summary.bank_id -> banks.bank_id
  OK  ghg_methodology.bank_id -> banks.bank_id
  OK  governance.bank_id -> banks.bank_id
  OK  internal_carbon_price.bank_id -> banks.bank_id
  OK  investments.bank_id -> banks.bank_id
  OK  physical_risk_exposures.bank_id -> banks.bank_id
  OK  rec_registry.bank_id -> banks.bank_id
  OK  resilience_assessment.bank_id -> banks.bank_id
  OK  scope12_consolidation.bank_id -> banks.bank_id
  OK  scope3_categories

## Write patched dataset + provenance

In [6]:
for name, df in tables.items():
    df.to_csv(os.path.join(PATCHED_DIR, name + ".csv"), index=False)
with open(os.path.join(PATCHED_DIR, "_patch_log.json"), "w") as f:
    json.dump({"source_dir": RAW_DIR,
               "tables_written": sorted(tables),
               "dropped_tables": sorted(DROP_TABLES),
               "vehicle_scope1_recalc": True,
               "gov_snap_applied": APPLY_GOV_SNAP,
               "gov_fixes": fix_log,
               "warnings": warnings}, f, indent=2, default=str)
print(f"{len(tables)} tables -> {PATCHED_DIR}/  (+ _patch_log.json)")

30 tables -> C:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data_patched/  (+ _patch_log.json)
